# **1. Perkenalan Dataset**


## Telco Customer Churn Dataset

**Sumber Dataset**: [Kaggle - Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

**Deskripsi**:
Dataset ini berisi informasi pelanggan perusahaan telekomunikasi yang digunakan untuk memprediksi apakah seorang pelanggan akan berhenti berlangganan (churn) atau tidak.

**Karakteristik Dataset**:
- **Jumlah Baris**: 7.043 pelanggan
- **Jumlah Kolom**: 21 fitur
- **Target Variable**: `Churn` (Yes/No — Binary Classification)
- **Tipe Data**: Campuran (Numerik & Kategorikal)

**Fitur-fitur Utama**:
| Fitur | Deskripsi | Tipe |
|---|---|---|
| customerID | ID unik pelanggan | String |
| gender | Jenis kelamin | Kategorikal |
| SeniorCitizen | Apakah warga senior (0/1) | Numerik |
| Partner | Memiliki pasangan | Kategorikal |
| Dependents | Memiliki tanggungan | Kategorikal |
| tenure | Lama berlangganan (bulan) | Numerik |
| PhoneService | Layanan telepon | Kategorikal |
| MultipleLines | Multiple lines | Kategorikal |
| InternetService | Jenis layanan internet | Kategorikal |
| OnlineSecurity | Keamanan online | Kategorikal |
| OnlineBackup | Backup online | Kategorikal |
| DeviceProtection | Proteksi perangkat | Kategorikal |
| TechSupport | Dukungan teknis | Kategorikal |
| StreamingTV | Streaming TV | Kategorikal |
| StreamingMovies | Streaming film | Kategorikal |
| Contract | Jenis kontrak | Kategorikal |
| PaperlessBilling | Tagihan tanpa kertas | Kategorikal |
| PaymentMethod | Metode pembayaran | Kategorikal |
| MonthlyCharges | Biaya bulanan | Numerik |
| TotalCharges | Total biaya | Numerik |
| Churn | Target (Yes/No) | Kategorikal |

**Tujuan**: Membangun model machine learning untuk memprediksi customer churn, sehingga perusahaan dapat mengambil tindakan preventif untuk mempertahankan pelanggan.

**Author**: Eko Muhammad Rizki  
**Project**: SMSML_Eko_Muhammad_Rizki

# **2. Import Library**

Pada tahap ini, kita mengimpor semua pustaka (library) Python yang dibutuhkan untuk analisis data, visualisasi, preprocessing, dan pembangunan model machine learning.

In [ ]:
# Install dependencies (jika di Google Colab)
# !pip install pandas numpy scikit-learn matplotlib seaborn mlflow

# === Data Manipulation ===
import pandas as pd
import numpy as np

# === Visualization ===
import matplotlib.pyplot as plt
import seaborn as sns

# === Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# === Models ===
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# === Evaluation ===
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve
)

# === Utilities ===
import warnings
warnings.filterwarnings('ignore')

# === Display settings ===
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

# **3. Memuat Dataset**

Pada tahap ini, kita memuat dataset Telco Customer Churn dari URL publik IBM Watson. Dataset dalam format CSV dan langsung dibaca menggunakan pandas.

In [ ]:
# === Load Dataset ===
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(f'Dataset berhasil dimuat!')
print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print(f'\nNama Kolom:')
print(list(df.columns))

In [ ]:
# === Menampilkan 5 baris pertama ===
df.head()

In [ ]:
# === Informasi Dataset ===
df.info()

In [ ]:
# === Statistik Deskriptif ===
df.describe(include='all')

In [ ]:
# === Cek Missing Values ===
print('Missing values per kolom:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# === Cek data unik dan tipe ===
for col in df.columns:
    print(f'{col}: {df[col].dtype} — {df[col].nunique()} unique values')
    if df[col].dtype == 'object' and df[col].nunique() < 10:
        print(f'  Values: {df[col].value_counts().to_dict()}')

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# === 4.1 Distribusi Target Variable (Churn) ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribusi Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Jumlah Pelanggan')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0, 0.05],
            shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Persentase Churn', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Churn ratio: {df["Churn"].value_counts(normalize=True).to_dict()}')

In [ ]:
# === 4.2 Distribusi Fitur Numerik ===
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=axes[idx],
                 palette=['#2ecc71', '#e74c3c'], alpha=0.6)
    axes[idx].set_title(f'Distribusi {col} by Churn', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# === 4.3 Boxplot Fitur Numerik ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[idx],
                palette=['#2ecc71', '#e74c3c'])
    axes[idx].set_title(f'{col} vs Churn', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# === 4.4 Distribusi Fitur Kategorikal ===
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[idx],
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[idx].set_title(f'{col} vs Churn (%)', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Percentage')
    axes[idx].legend(title='Churn', loc='upper right')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# === 4.5 Correlation Heatmap ===
# Encode for correlation analysis
df_encoded = df.copy()
df_encoded['TotalCharges'] = pd.to_numeric(df_encoded['TotalCharges'], errors='coerce')

# Label encode binary columns for correlation
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Select only numeric columns
numeric_df = df_encoded.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 10))
correlation = numeric_df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            square=True, vmin=-1, vmax=1)
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Show correlation with target
print('\nKorelasi dengan Churn:')
churn_corr = correlation['Churn'].sort_values(ascending=False)
print(churn_corr)

In [ ]:
# === 4.6 Tenure vs Churn Analysis ===
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Tenure distribution by churn
sns.kdeplot(data=df[df['Churn'] == 'No'], x='tenure', ax=axes[0],
            shade=True, color='#2ecc71', label='No Churn')
sns.kdeplot(data=df[df['Churn'] == 'Yes'], x='tenure', ax=axes[0],
            shade=True, color='#e74c3c', label='Churn')
axes[0].set_title('Tenure Distribution by Churn', fontsize=12, fontweight='bold')
axes[0].legend()

# Monthly Charges vs Total Charges scatter
scatter_data = df.dropna(subset=['TotalCharges'])
colors_map = {'Yes': '#e74c3c', 'No': '#2ecc71'}
for label, color in colors_map.items():
    mask = scatter_data['Churn'] == label
    axes[1].scatter(scatter_data[mask]['MonthlyCharges'],
                    scatter_data[mask]['TotalCharges'],
                    c=color, alpha=0.3, label=label, s=10)
axes[1].set_title('Monthly vs Total Charges by Churn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Monthly Charges')
axes[1].set_ylabel('Total Charges')
axes[1].legend(title='Churn')

plt.tight_layout()
plt.show()

In [ ]:
# === 4.7 Ringkasan EDA ===
print('='*60)
print('RINGKASAN EDA - TELCO CUSTOMER CHURN')
print('='*60)
print(f'\n1. Dataset memiliki {df.shape[0]} baris dan {df.shape[1]} kolom')
print(f'2. Target variable (Churn) tidak seimbang:')
print(f'   - No Churn: {(df["Churn"]=="No").sum()} ({(df["Churn"]=="No").mean()*100:.1f}%)')
print(f'   - Churn: {(df["Churn"]=="Yes").sum()} ({(df["Churn"]=="Yes").mean()*100:.1f}%)')
print(f'3. TotalCharges memiliki {df["TotalCharges"].isna().sum()} missing values')
print(f'4. Pelanggan dengan tenure pendek cenderung churn')
print(f'5. Contract month-to-month memiliki churn rate tertinggi')
print(f'6. Pelanggan dengan MonthlyCharges tinggi cenderung churn')
print(f'7. Fiber optic memiliki churn rate lebih tinggi dari DSL')

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Tahapan yang dilakukan:
1. Menghapus kolom yang tidak relevan (customerID)
2. Menangani missing values (TotalCharges)
3. Menghapus data duplikat
4. Feature Engineering (tenure_group, charge_ratio, dll)
5. Encoding data kategorikal (LabelEncoder + OneHotEncoder)
6. Train/Test Split (80/20 stratified)
7. Normalisasi fitur numerik (StandardScaler)

In [ ]:
# === 5.1 Hapus Kolom Tidak Relevan ===
df_clean = df.copy()

# Drop customerID
df_clean = df_clean.drop(columns=['customerID'])
print(f'Kolom customerID dihapus. Remaining: {df_clean.shape[1]} kolom')

In [ ]:
# === 5.2 Handle Missing Values ===
# TotalCharges: convert to numeric, fill NaN with median
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

missing_count = df_clean['TotalCharges'].isna().sum()
print(f'Missing values di TotalCharges: {missing_count}')

median_total = df_clean['TotalCharges'].median()
df_clean['TotalCharges'].fillna(median_total, inplace=True)
print(f'Filled with median: {median_total:.2f}')
print(f'Remaining missing: {df_clean.isnull().sum().sum()}')

In [ ]:
# === 5.3 Hapus Duplikat ===
duplicates = df_clean.duplicated().sum()
print(f'Jumlah duplikat: {duplicates}')
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'Duplikat dihapus. Remaining: {df_clean.shape[0]} baris')

In [ ]:
# === 5.4 Feature Engineering ===

# Tenure groups
bins = [0, 12, 24, 48, 60, np.inf]
labels = ['0-12', '13-24', '25-48', '49-60', '61+']
df_clean['tenure_group'] = pd.cut(df_clean['tenure'], bins=bins, labels=labels)

# Charge per month ratio
df_clean['charge_per_month_ratio'] = df_clean['TotalCharges'] / (df_clean['tenure'] + 1)

# Is new customer
df_clean['is_new_customer'] = (df_clean['tenure'] <= 6).astype(int)

# Internet add-on count
addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']
df_clean['has_internet_addon'] = df_clean[addon_cols].apply(
    lambda row: sum(1 for val in row if val == 'Yes'), axis=1
)

# Monthly charge category
charge_bins = [0, 30, 60, 90, np.inf]
charge_labels = ['Low', 'Medium', 'High', 'Very High']
df_clean['monthly_charge_category'] = pd.cut(
    df_clean['MonthlyCharges'], bins=charge_bins, labels=charge_labels
)

print('Feature Engineering selesai!')
print(f'Fitur baru: tenure_group, charge_per_month_ratio, is_new_customer, has_internet_addon, monthly_charge_category')
print(f'Total kolom sekarang: {df_clean.shape[1]}')

In [ ]:
# === 5.5 Encoding Kategorikal ===

# Label Encoding untuk kolom binary
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f'Label Encoded: {col}')

# One-Hot Encoding untuk multi-category columns
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                  'OnlineBackup', 'DeviceProtection', 'TechSupport',
                  'StreamingTV', 'StreamingMovies', 'Contract',
                  'PaymentMethod', 'tenure_group', 'monthly_charge_category']

df_clean = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True, dtype=int)
print(f'\nOne-Hot Encoding selesai!')
print(f'Total fitur setelah encoding: {df_clean.shape[1]}')

In [ ]:
# === 5.6 Train/Test Split ===
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')
print(f'Churn ratio - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}')

In [ ]:
# === 5.7 Normalisasi/Scaling ===
numerical_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_per_month_ratio']

scaler = StandardScaler()
X_train[numerical_to_scale] = scaler.fit_transform(X_train[numerical_to_scale])
X_test[numerical_to_scale] = scaler.transform(X_test[numerical_to_scale])

print('StandardScaler applied!')
print(f'\nScaled columns: {numerical_to_scale}')
print(f'\nX_train sample (first 5 rows):')
X_train.head()

In [ ]:
# === 5.8 Ringkasan Preprocessing ===
print('='*60)
print('RINGKASAN PREPROCESSING')
print('='*60)
print(f'\n1. Kolom customerID dihapus')
print(f'2. {missing_count} missing values di TotalCharges diisi dengan median')
print(f'3. {duplicates} duplikat ditemukan')
print(f'4. 5 fitur baru dibuat (feature engineering)')
print(f'5. Binary columns: Label Encoded')
print(f'6. Multi-category columns: One-Hot Encoded')
print(f'7. Train/Test split: 80/20 (stratified)')
print(f'8. Numerical columns: StandardScaler applied')
print(f'\nFinal dataset shape:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}')
print(f'  y_test: {y_test.shape}')
print(f'\nFeature names ({len(X_train.columns)}):')
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i}. {col}')